# AEGIS-SQL — Colab에서 3분 만에 재현하기

한국 금융·보험사의 레거시 스키마 위에서 동작하는 **거버넌스 내장형 Text-to-SQL 엔진**입니다.
이 노트북은 README가 게시한 측정 결과를 이 런타임에서 다시 만들어 냅니다.

**API 키가 필요 없습니다.** LLM 티어가 없으면 결정론 template 티어로 폴백합니다 —
CI에서도 같은 경로로 전체 파이프라인이 돕니다. **GPU도 필요 없습니다.** 기본 CPU 런타임 그대로 두세요.

| 단계 | 소요 |
|---|---|
| 클론 + 설치 | 약 1분 |
| 데모 DB 생성(37만 행) + 벤치마크 gold SQL 실행 검증 | 약 10초 |
| 거버넌스 3막 시연 | 약 5초 |
| **벤치마크 106문항 + 어블레이션 11변형** | 약 30초 |
| 테스트 252개 | 약 10초 |

### 무엇을 보면 되는가

1. **EX 44.4%** 가 [README 표](https://github.com/sokldjs554/aegis-sql#측정-결과)와 같은가
2. 주민등록번호 요청이 **SQL이 만들어지기 전에** 막히는가
3. 스키마 지문이 **`26cee9e1989d6426`** 인가

> 코드 셀의 출력은 비워 두었습니다. 재현 가능한 것을 미리 채워 두면
> '눌러서 돈다'는 이 노트북의 전제를 스스로 갉아먹기 때문입니다.
> 대신 각 셀 위에 **예상 출력**을 적어 두었으니 화면과 대조하세요.
> 여기서 재현할 수 **없는** 것(LLM 티어 실측)만 저장된 리포트로 보여줍니다.

*"이 노트북은 Google이 작성하지 않았습니다" 경고가 뜨면 **[무시하고 계속]**을 누르세요.*

저장소: <https://github.com/sokldjs554/aegis-sql>

---
## 1. 클론과 설치

설치되는 것은 코어 의존성뿐입니다. PyTorch도 TensorFlow도 받지 않습니다 —
서빙 경로는 numpy 추론만 쓰기 때문입니다.

**빨간 경고 한 줄이 뜹니다.** Colab의 `ibis-framework`가 옛 sqlglot을 요구하는데
이 프로젝트는 27.4 이상이 필요해서입니다. 이 노트북은 ibis를 쓰지 않으므로 무시해도 됩니다.

> 예상 출력 → `Successfully installed aegis-sql-0.1.0 ...`

> 설치 후 **런타임을 재시작할 필요는 없습니다.** 다음 셀이 알아서 경로를 잡습니다.

In [ ]:
!git clone --depth 1 https://github.com/sokldjs554/aegis-sql.git /content/aegis-sql
!pip install -q -e /content/aegis-sql pytest-asyncio

### 설치 검증과 환경 설정

`aegis version`이 프로바이더 가용성을 그대로 찍습니다 —
**키가 없다는 사실 자체를 화면에 남기려는 것**입니다.

> 예상 출력 → `sqlglot 30.x · aegis_sql import OK` 와
> `providers {'anthropic': False, 'openai': False, 'mock': True}`

In [ ]:
import os, sys

REPO = '/content/aegis-sql'
os.chdir(REPO)
os.environ['COLUMNS'] = '120'   # rich 가 비-TTY 에서 80칸으로 줄바꿈하는 것을 넓힌다

# editable 설치는 site-packages 에 .pth 를 남기는데, 그 파일은 인터프리터가
# '시작할 때'만 읽힌다. 이미 돌고 있는 Colab 커널은 방금 설치한 것을 모른다.
# 런타임을 재시작하는 대신 소스 경로를 직접 넣는다 (이 저장소는 src 레이아웃이다).
# 셸 셀의 `!aegis ...` 는 매번 새 프로세스라 원래부터 영향이 없다.
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')

import sqlglot, aegis_sql
from sqlglot import expressions as exp
assert hasattr(exp, 'Attach'), 'sqlglot 이 너무 낮습니다 — ATTACH 가드를 만들 수 없습니다'
print('sqlglot', sqlglot.__version__, '· aegis_sql', aegis_sql.__version__, '· import OK')

# 노트북의 나머지는 전부 `!aegis ...` 로 도므로, 그 경로도 함께 확인한다.
!python -c "import aegis_sql, sqlglot; print('서브프로세스에서도 OK')"

In [ ]:
!aegis version

---
## 2. 데이터를 만듭니다

데모 DB는 저장소에 들어 있지 않습니다. **시드된 생성기로 지금 만듭니다** —
그래서 누가 언제 돌려도 같은 데이터가 나오고, 벤치마크의 정답 SQL 결과가 재현됩니다.

이어서 벤치마크 106문항의 **정답 SQL을 방금 만든 DB에서 실제로 실행**해 검증합니다.
실행되지 않는 정답이 하나라도 있으면 빌드가 실패합니다 — 벤치마크가 종이 위 숫자가 아니라는 뜻입니다.

> 예상 출력 → `373,778 rows` 규모의 11개 테이블, 그리고 `106 items`

In [ ]:
!python scripts/build_demo_db.py
!python scripts/build_benchmark.py

---
## 3. 이 엔진이 끝나는 세 가지 방식

`aegis demo`가 대표 질의를 한 번에 보여줍니다.

- **● 조회** — 스키마 링킹부터 실행까지, 어느 테이블을 왜 골랐는지와 함께
- **■ 차단** — 개인정보·DDL 요청은 SQL이 만들어지기 **전에** 거부
- **◆ 되묻기** — 기준이 모호하면 그럴듯한 숫자를 지어내지 않고 먼저 되묻습니다

`--log-level ERROR`로 로그를 껐습니다. 가드가 실제로 막은 기록은 다음 절에서 봅니다.

> 예상 출력 → `● ok` / `■ blocked` / `◆ clarify` 배지와 각각의 SQL·사유

In [ ]:
!aegis demo --log-level ERROR

---
## 4. 거버넌스는 프롬프트가 아니라 AST에 있습니다

여기가 이 프로젝트의 핵심입니다. 아래는 **엔진이 만든 SQL이 아니라 여러분이 넣은 임의의 SQL**을
실행하지 않고 정책만 통과시킵니다.

`SELECT *`를 던지면 별칭까지 확장 추적해서 반출 금지 컬럼을 찾아내고, 마스킹과 LIMIT을 주입합니다.
프롬프트에 "개인정보를 조회하지 마세요"라고 적는 방식과는 다릅니다 — 우회가 불가능합니다.

> 예상 출력 → `PII_FORBIDDEN [TB_CUST.RRNO_ENC]` 차단 + `mask:` 재작성 + `limit-injected:200`

In [ ]:
!aegis policy "SELECT * FROM TB_CUST"

### 세션 컨텍스트가 행 수준 정책을 켭니다

같은 질문이라도 **누가 어떤 목적으로 묻느냐에 따라 실행되는 SQL이 달라집니다.**

> 예상 출력 → 첫 셀에 `WHERE TB_AGNT.BRCH_CD = 'BR003'`,
> 둘째 셀에 `WHERE TB_CUST.MKT_AGR_YN = 'Y'` (개인정보보호법 제22조)

In [ ]:
!aegis policy "SELECT AGNT_NM FROM TB_AGNT" --branch BR003
!aegis policy "SELECT COUNT(*) FROM TB_CUST" --purpose marketing

---
## 5. 학습은 Keras, 서빙은 numpy

캐스케이드 라우터는 TensorFlow로 학습하고 **numpy 가중치로 export**합니다.
서빙 경로에 TensorFlow 의존성이 0이라는 뜻입니다 — 지금 이 노트북에도 안 깔려 있습니다.

아래 로그에서 `router loaded ... auc=0.9696`을 확인하세요.

> 예상 출력 → `cascade router loaded path=.../models/router` 와
> `engine ready tables=11 tiers=['template']`

In [ ]:
!aegis ask "작년 하반기에 체결된 계약 중 월납보험료가 20만원 이상인 건수를 지점별로 알려줘" \
    --explain --log-level INFO 2>&1 | grep -E 'router loaded|engine ready|난이도|신뢰도' | head -20

---
## 6. 벤치마크 106문항 전부 + 어블레이션 11변형

문항을 자르지 않습니다. 전체가 30초 안에 끝나므로 자를 이유가 없습니다.

**어블레이션이 이 프로젝트의 전제를 스스로 검증합니다** — 구성요소를 하나씩 빼면서
정확도가 얼마나 떨어지는지 봅니다. 41종 사내 용어사전을 빼면 −10.0%p입니다.
모델 크기가 아니라 **도메인 지식 주입**이 이 도메인의 정확도를 가른다는 주장이,
저자의 의견이 아니라 표의 한 줄로 나옵니다.

> 예상 출력 → `EX 44.4%` · easy 90.0% · medium 32.5% · hard 0.0%
> · 거버넌스 10/10 · 모호성 6/6 · **비용 $0**

In [ ]:
!aegis eval --ablation --report reports/eval.md --no-failures --log-level ERROR

### 재현 정보 — 이게 클라이맥스입니다

방금 만든 리포트의 푸터입니다. **스키마 지문이 README의 `26cee9e1989d6426`과 같은지** 보세요.

이 프로젝트의 설계 원칙 4번은 *"재현되지 않는 숫자는 측정이 아니다"*입니다.
같은 질문에 같은 답이 나오는지는 **스키마 지문 · 프롬프트 버전 · 생성 티어** 세 값이
같은지로 판정합니다.

In [ ]:
import json, re
text = open('reports/eval.md', encoding='utf-8').read()
m = re.search(r'```json\s*(\{.*?\})\s*```', text, re.S)
print(json.dumps(json.loads(m.group(1)), ensure_ascii=False, indent=2) if m else text[-1500:])

---
## 7. 테스트 — 목킹 없이 실제 DB 대상

학습 테스트(`slow` 마커)를 제외한 252개입니다.

> 예상 출력 → `252 passed`

In [ ]:
!pytest -q -m "not slow" 2>&1 | tail -5

---
## 8. 웹 콘솔 (선택)

여기까지로 이 프로젝트의 주장은 전부 확인됐습니다. 이 절은 **제품을 직접 만져보는** 부분입니다.

Colab 세션이 살아 있는 동안만 동작합니다. iframe이 비어 보이면 아래 폴백 셀로 확인하세요.

In [ ]:
import subprocess, sys, time, requests

proc = subprocess.Popen(
    [sys.executable, '-m', 'aegis_sql.cli', 'serve', '--port', '8000', '--log-level', 'WARNING'],
    cwd=REPO,
)
for _ in range(120):
    try:
        if requests.get('http://127.0.0.1:8000/v1/health', timeout=1).ok:
            print('엔진 준비됨'); break
    except Exception:
        time.sleep(0.5)
else:
    print('기동 실패 — 아래 셀들은 건너뛰세요')

**콘솔에서 해볼 것** — 상단 `목적`을 `마케팅`으로 바꾼 뒤 세 번째 질문을 던지면
SQL에 `MKT_AGR_YN = 'Y'`가 저절로 붙는 것을 볼 수 있습니다.

1. `작년 하반기에 체결된 계약 중 월납보험료가 20만원 이상인 건수를 지점별로 알려줘`
2. `고객 이름이랑 주민등록번호 좀 뽑아줘`
3. `마케팅 수신 대상 고객 수를 성별로 알려줘`

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)

**폴백** — iframe이 비어 보여도 엔진은 돌고 있습니다. 이 셀이 그걸 증명합니다.

In [ ]:
r = requests.post('http://127.0.0.1:8000/v1/query',
                  json={'question': '고객 이름이랑 주민등록번호 좀 뽑아줘', 'explain': True}).json()
print('상태   :', r['status'])
for v in r.get('violations', []):
    print('위반   :', v['code'], '·', v['subject'])
    print('        ', v['message'])

---
## 여기서 재현되지 않는 것

정직하게 적어 둡니다.

- **LLM 티어 실측은 여기서 안 됩니다.** API 키가 필요합니다.
  아래 셀이 저장소에 커밋된 실측 리포트를 렌더합니다 — claude-sonnet-5로 돌린 결과입니다.
- **자체 학습 sLLM(5.3M)의 EX는 0.0%입니다.** 승격 규칙에 따라 캐스케이드에서 빠져 있습니다.
  `docs/SLM.md`에 왜 그런지 적어 두었습니다.
- **벤치마크 106문항은 작습니다.** Spider·BIRD 급 규모가 아닙니다.

본인 키로 캐스케이드를 켜보고 싶으시면:
`!pip install -q -e "/content/aegis-sql[llm]"` 후
`os.environ['ANTHROPIC_API_KEY']='...'` 그리고 `!aegis eval --limit 20`

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/eval_llm_only.md', encoding='utf-8').read()))

---
## 다음에 볼 것

| 문서 | 내용 |
|---|---|
| [`docs/GOVERNANCE.md`](https://github.com/sokldjs554/aegis-sql/blob/main/docs/GOVERNANCE.md) | 컬럼 4등급 · 마스킹 · 행 정책 · k-익명성 |
| [`docs/EVALUATION.md`](https://github.com/sokldjs554/aegis-sql/blob/main/docs/EVALUATION.md) | 측정 방법과 **알고 있는 한계** |
| [`docs/FLYWHEEL.md`](https://github.com/sokldjs554/aegis-sql/blob/main/docs/FLYWHEEL.md) | 스키마만으로 12,540쌍 |

저장소: <https://github.com/sokldjs554/aegis-sql>